# 14 — Graph Nodes & Full Pipeline (`graph/nodes.py`, `graph/graph.py`)
This notebook walks through each LangGraph node individually, then runs the complete graph.

**Node sequence:**
```
START → pre_hook → supervisor → [agents in parallel] → auto_ticket → synthesizer → post_hook → END
```

Short-circuit: if `guardrail_passed=False`, jumps from `pre_hook` → `post_hook` → `END`.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'codebase', 'codebase', 'src'))
os.makedirs("logs", exist_ok=True)
os.environ["ENABLE_MOCK"] = "true"
os.environ["USE_MCP"] = "false"
# Set dummy env vars so agents don't raise EnvironmentError
os.environ.setdefault("DATABRICKS_HOST", "fake-host")
os.environ.setdefault("DATABRICKS_TOKEN", "fake-token")
os.environ.setdefault("DATABRICKS_HTTP_PATH", "fake-path")
os.environ.setdefault("COLLIBRA_BASE_URL", "http://fake-collibra")
os.environ.setdefault("COLLIBRA_API_TOKEN", "fake-token")
os.environ.setdefault("JIRA_BASE_URL", "http://fake-jira")
os.environ.setdefault("JIRA_EMAIL", "fake@fake.com")
os.environ.setdefault("JIRA_API_TOKEN", "fake-token")

## 1. pre_hook_node — Guardrails + Timer

In [ ]:
from graph.nodes import pre_hook_node
from graph.state import initial_state

state = initial_state("What is the GRR for retention last month?")
result = pre_hook_node(state)
print("guardrail_passed:", result["guardrail_passed"])
print("query (cleaned) :", result.get("query", state["query"]))
print("start_time set  :", result["start_time"] > 0)

In [ ]:
# Blocked query
blocked_state = initial_state("DROP TABLE analytics.retention_metrics")
result = pre_hook_node(blocked_state)
print("guardrail_passed:", result["guardrail_passed"])
print("final_summary   :", result["final_summary"])

## 2. supervisor_node — Intent Classification + Routing

In [ ]:
from graph.nodes import supervisor_node

state = initial_state("What is the GRR for retention last month?")
state["guardrail_passed"] = True
result = supervisor_node(state)
print("intent      :", result["intent"])
print("next_agents :", result["next_agents"])
print("data_products:", result["data_products"])
print("confidence  :", result["confidence"])

## 3. Route-After-Pre-Hook Conditional

In [ ]:
from graph.graph import route_after_pre_hook

# Normal flow
state1 = {"guardrail_passed": True}
print("Normal:", route_after_pre_hook(state1))   # "supervisor"

# Blocked
state2 = {"guardrail_passed": False}
print("Blocked:", route_after_pre_hook(state2))  # "post_hook" 

## 4. Route-To-Agents Fan-Out

In [ ]:
from graph.graph import route_to_agents

state = {
    "next_agents": ["information", "knowledge"],
    "query": "What is GRR?",
    "data_products": ["retention"],
    "time_range": "last_month",
}
sends = route_to_agents(state)
print("Fan-out type:", type(sends))
if isinstance(sends, list):
    for s in sends:
        print(f"  Send → node={s.node} state_keys={list(s.arg.keys())[:4]}")

## 5. rule_node — Direct Call

In [ ]:
from graph.nodes import rule_node

state = initial_state("list rules for retention")
state["intent"] = "write_rule"
state["data_products"] = ["retention"]

result = rule_node(state)
print("agent_results count:", len(result["agent_results"]))
print("success:", result["agent_results"][0]["success"])
print("summary (first 200):", result["agent_results"][0]["summary"][:200])

## 6. synthesizer_node — String Fallback (no LLM key)

In [ ]:
from graph.nodes import synthesizer_node

state = initial_state("What is GRR?")
state.update({
    "intent": "knowledge_lookup",
    "data_products": ["retention"],
    "confidence": 0.85,
    "auto_tickets": [],
    "agent_results": [
        {"agent": "rule_agent", "success": True, "summary": "GRR >= 85% threshold rule active.", "confidence": 1.0},
    ],
})
result = synthesizer_node(state)
print("final_summary:", result["final_summary"][:300])
print("confidence   :", result["confidence"])

## 7. post_hook_node — Timing + Audit Log

In [ ]:
import time
from graph.nodes import post_hook_node

state = initial_state("test")
state.update({
    "start_time": time.time() - 0.35,  # 350ms ago
    "guardrail_passed": True,
    "intent": "knowledge_lookup",
    "confidence": 0.88,
    "agent_results": [{"agent": "rule_agent"}],
})
result = post_hook_node(state)
print("execution_ms:", result["execution_ms"])

## 8. Full Graph — End-to-End Run

In [ ]:
from graph.graph import copilot_graph
from graph.state import initial_state

state = initial_state(
    query="What are the rules for data quality?",
    thread_id="nb-demo-01",
    user_id="notebook_user",
)

print("Running graph...")
try:
    final_state = copilot_graph.invoke(
        state,
        config={"configurable": {"thread_id": "nb-demo-01"}},
    )
    print("\n=== RESULT ===")
    print("intent       :", final_state.get("intent"))
    print("confidence   :", final_state.get("confidence"))
    print("execution_ms :", final_state.get("execution_ms"))
    print("agents used  :", [r["agent"] for r in final_state.get("agent_results", [])])
    print("\nFinal Summary:")
    print(final_state.get("final_summary", "")[:500])
except Exception as e:
    print("Error (expected in mock mode without live services):", type(e).__name__, str(e)[:200])